# OPD-study Colab quickstart

Run a CPU-safe SFT/KD/OPD plumbing comparison, create checkpoints, a static report, and TensorBoard logs. A smoke run verifies the path; remove `smoke=True` for a longer toy experiment. No Qwen weights are downloaded.

In [1]:
import sys
IN_COLAB = 'google.colab' in sys.modules
print({'in_colab': IN_COLAB, 'python': sys.version.split()[0]})

{'in_colab': False, 'python': '3.12.13'}


In [2]:
import subprocess
from pathlib import Path
if IN_COLAB:
    repo_root = Path('/content/OPD-study')
    if not (repo_root / '.git').exists():
        subprocess.check_call(['git', 'clone', '--depth', '1',
            'https://github.com/BangProx/OPD-study.git', str(repo_root)])
    subprocess.check_call([sys.executable, '-m', 'pip', 'install',
        '-e', str(repo_root)])
else:
    repo_root = Path.cwd()
    if not (repo_root / 'src').exists():
        repo_root = Path.cwd().parents[1]
    print('Local validation: using repository src/; no network call.')
# An editable install writes a .pth hook that this already-running Colab kernel
# does not process until restart, so make the freshly cloned source importable now.
sys.path.insert(0, str(repo_root / 'src'))

Local validation: using repository src/; no network call.


In [3]:
from opd_study.demo import run_demo
print('OPD-study imported')

OPD-study imported


In [4]:
output = Path('/content/opd-study-demo' if IN_COLAB else 'artifacts/colab-local')
summary = run_demo(output, smoke=True, requested_device='cpu')
print({name: {'loss': round(run['evaluation']['loss'], 4),
              'tokens': run['response_tokens']}
       for name, run in summary['runs'].items()})

Matplotlib is building the font cache; this may take a moment.


{'no_train': {'loss': 4.4078, 'tokens': 0}, 'sft': {'loss': 4.3832, 'tokens': 4}, 'off_policy_kd': {'loss': 4.3992, 'tokens': 4}, 'opd': {'loss': 4.4024, 'tokens': 4}}


In [5]:
required = {'summary.json', 'experiment-card.json', 'index.html',
            'loss_curves.png', 'distribution_diagnostics.png'}
assert required.issubset({path.name for path in output.iterdir()})
assert summary['fairness']['same_initial_student']
print('Report:', output / 'index.html')
print('TensorBoard:', output / 'tensorboard')

Report: artifacts/colab-local/index.html
TensorBoard: artifacts/colab-local/tensorboard


## Optional CUDA LoRA and QLoRA smoke (manual opt-in)

The default above downloads no model. The cell below stays disabled. Enabling it accepts the MIT GSM8K and Apache-2.0 Qwen terms and permits about 5.57GB of pinned weights plus 2.73MB of data. Use a CUDA runtime; each successful one-step update is a plumbing check, not a benchmark result. Run LoRA first, then QLoRA; both subprocesses reuse the same download cache.

In [6]:
RUN_OPTIONAL_QWEN_LORA = False
lora_output = Path('/content/opd-study-qwen-lora')
if RUN_OPTIONAL_QWEN_LORA:
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError('Select a CUDA runtime before LoRA.')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install',
        '-e', f'{repo_root}[research]'])
    subprocess.check_call([sys.executable, '-m', 'opd_study',
        'research-train', '--config',
        str(repo_root / 'configs/laptop/gsm8k_lora.yaml'),
        '--cache', '/content/opd-study-cache', '--output', str(lora_output),
        '--smoke', '--accept-dataset-license', '--accept-model-license'],
        cwd=repo_root)
else:
    print('SKIPPED: set RUN_OPTIONAL_QWEN_LORA=True only after review.')

SKIPPED: set RUN_OPTIONAL_QWEN_LORA=True only after review.


In [7]:
if RUN_OPTIONAL_QWEN_LORA:
    assert (lora_output / 'experiment-card.json').is_file()
    assert (lora_output / 'metrics.jsonl').is_file()
    assert (lora_output / 'adapter').is_dir()
    assert (lora_output / 'checkpoints/optimizer.pt').is_file()
    assert any((lora_output / 'tensorboard').iterdir())
    print('LoRA update/save/eval/TensorBoard artifact contract passed.')
else:
    print('LoRA smoke remained disabled by default.')

LoRA smoke remained disabled by default.


In [8]:
RUN_OPTIONAL_QWEN_QLORA = False
qlora_output = Path('/content/opd-study-qwen-qlora')
if RUN_OPTIONAL_QWEN_QLORA:
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError('Select a CUDA runtime before QLoRA.')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install',
        '-e', f'{repo_root}[research,qlora]'])
    subprocess.check_call([sys.executable, '-m', 'opd_study',
        'research-train', '--config',
        str(repo_root / 'configs/laptop/gsm8k_qlora.yaml'),
        '--cache', '/content/opd-study-cache', '--output', str(qlora_output),
        '--smoke', '--accept-dataset-license', '--accept-model-license'],
        cwd=repo_root)
else:
    print('SKIPPED: set RUN_OPTIONAL_QWEN_QLORA=True only after review.')

SKIPPED: set RUN_OPTIONAL_QWEN_QLORA=True only after review.


In [9]:
if RUN_OPTIONAL_QWEN_QLORA:
    assert (qlora_output / 'experiment-card.json').is_file()
    assert (qlora_output / 'metrics.jsonl').is_file()
    assert (qlora_output / 'adapter').is_dir()
    assert (qlora_output / 'checkpoints/optimizer.pt').is_file()
    assert any((qlora_output / 'tensorboard').iterdir())
    print('QLoRA update/save/eval/TensorBoard artifact contract passed.')
else:
    print('QLoRA smoke remained disabled by default.')

QLoRA smoke remained disabled by default.


In [10]:
if not RUN_OPTIONAL_QWEN_LORA and not RUN_OPTIONAL_QWEN_QLORA:
    print('Default Colab path remained CPU-safe and model-download-free.')
else:
    print('Selected CUDA smoke paths completed.')

Default Colab path remained CPU-safe and model-download-free.


## Next

Open the generated `index.html`, then continue with Korean or English Lesson 00. This notebook validates Colab only after the repository has been published; local stored output is not evidence of a hosted-runtime run.